# clef-flash-tpu-int8: ARC-Challenge accuracy and latency

Checks the published TPU port against Cloudflare's own numbers, on one benchmark from the Decision Index that the [clef-flash model card](https://huggingface.co/Cloudflare/clef-flash) reports:
- **ARC-Challenge:** 1,172 grade-school science questions with 3 to 5 options. Cloudflare's BF16 clef-flash scores **98.3%**.
- The rows are built with the [Decision Index kit](https://github.com/apolinario/decision-index)'s own `choice_row`, from the ARC test split the kit uses, and run through the kit's runner, which validates every response.
- Both modes run: BF16 activations with int8 weights (the default), and int8 activations too (W8A8).

The last cell writes the results and tested versions into the repo's model card (`README.md`), between its `arc-challenge-results` markers, and uploads it. That needs an `HF_TOKEN` Colab secret with write access to the repo.

**Runtime:** Runtime > Change runtime type > **v5e-1 TPU**. Run the cells top to bottom.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

MODEL_REPO = "heisvibhor-1/clef-flash-tpu-int8"
KIT_URL = "https://github.com/apolinario/decision-index"
KIT_DIR = Path("/content/decision-index")

# joint_schema_model.py imports torch; only its prompt builder and answer format run here.
if importlib.util.find_spec("torch") is None:
    %pip install -q torch --index-url https://download.pytorch.org/whl/cpu
%pip install -q -U "transformers>=5.10" safetensors

if not KIT_DIR.exists():
    subprocess.run(["git", "clone", "-q", KIT_URL, str(KIT_DIR)], check=True)
KIT_COMMIT = subprocess.run(["git", "-C", str(KIT_DIR), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
%pip install -q {KIT_DIR}
print(f"decision-index @ {KIT_COMMIT}")

In [ ]:
import time

import jax
from huggingface_hub import HfApi, snapshot_download

# While the repo is private, add an HF_TOKEN Colab secret; huggingface_hub reads it.
MODEL_REVISION = HfApi().model_info(MODEL_REPO).sha
model_dir = snapshot_download(MODEL_REPO, revision=MODEL_REVISION, ignore_patterns=["compiled/*"])
sys.path.insert(0, model_dir)
import clef_tpu

started = time.perf_counter()
model = clef_tpu.load_model()
device_bytes = sum(array.nbytes for array in jax.tree.leaves([model.backbone, model.final_norm, model.head]))
DEVICE_KIND = jax.devices()[0].device_kind
print(f"{MODEL_REPO} @ {MODEL_REVISION}")
print(f"{device_bytes / 1e9:.1f} GB on {DEVICE_KIND} in {time.perf_counter() - started:.0f}s")

## ARC-Challenge rows

Built the way the kit's ARC builder (`normalize_direct.knowledge`) builds them: `choice_row` with the question as the instructions, the options keyed A to E, and an empty state. The kit builds ARC in the same function as MMLU, WinoGrande and HellaSwag, so its `suite rebuild --only 27` would need their sources too.

In [ ]:
import hashlib
import json

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

from decision_index.suite.build.layout import REFERENCE_MODEL, choice_row
from decision_index.suite.io import dumps

ARC_REPO = "allenai/ai2_arc"
ARC_FILE = "ARC-Challenge/test-00000-of-00001.parquet"
ARC_CATALOG_ID = 27
# The Decision Index's ARC-Challenge request count, after its exclusions.
ARC_REQUESTS = 1172
ROWS_PATH = Path("/content/arc-challenge-rows.jsonl")

ARC_REVISION = HfApi().dataset_info(ARC_REPO).sha
source_records = pq.read_table(hf_hub_download(ARC_REPO, ARC_FILE, repo_type="dataset", revision=ARC_REVISION)).to_pylist()


def arc_row(record):
    choices = record["choices"]
    row = choice_row("ARC-Challenge", "test", record["id"], record["question"], choices["text"], choices["label"].index(record["answerKey"]))
    payload = {"model": REFERENCE_MODEL, "state": row["state"], "questions": row["questions"]}
    row["_evaluation"] = {
        "run_id": f"{ARC_CATALOG_ID}:ARC-Challenge:{row['id']}",
        "catalog_id": ARC_CATALOG_ID,
        "dataset": "ARC-Challenge",
        "group_id": row["id"],
        "track": "ARC-Challenge",
        "payload_sha256": hashlib.sha256(dumps(payload).encode()).hexdigest(),
    }
    return row


arc_rows = [arc_row(record) for record in source_records]
assert len(arc_rows) == ARC_REQUESTS, f"{len(arc_rows)} rows, expected {ARC_REQUESTS}"
ROWS_PATH.write_text("".join(dumps(row) + "\n" for row in arc_rows))
expected_choices = {row["_evaluation"]["run_id"]: row["expected"]["q1"] for row in arc_rows}
print(f"{ARC_REPO} @ {ARC_REVISION}: {len(arc_rows)} requests")
print(json.dumps(arc_rows[0]["questions"], indent=2))

## Run

The kit's runner calls the engine once per row, checks each response (every option has a probability, they sum to 1, the choice is one of the options) and records its wall time. Warm-up first compiles the 1,024-token bucket and the head for 2 to 5 options, so no timed request includes compiling. Re-running the cell after a disconnect resumes where it stopped.

In [ ]:
from decision_index.engines import Engine
from decision_index.runner import run

RUNS_DIR = Path("/content/runs")
MODES = {"BF16 activations, int8 weights": False, "int8 activations (W8A8)": True}


class ClefTpuEngine(Engine):
    name = "clef-flash-tpu-int8"
    latency = "In-process request wall time on one TPU chip: prompt building, backbone, head and response; excludes loading and compiling."

    def __init__(self, int8_activations=False, **options):
        super().__init__(int8_activations=int8_activations, **options)
        self.int8_activations = int8_activations
        self.provenance = {"repo": MODEL_REPO, "revision": MODEL_REVISION, "int8_activations": int8_activations}

    def __call__(self, state, questions):
        request = {"model": self.name, "state": state, "questions": questions}
        return clef_tpu.systemone(model, request, self.int8_activations), None

    def warmup(self):
        # The head compiles once per option count; ARC questions have 3 to 5 options.
        for option_count in range(2, 6):
            criteria = {chr(65 + index): f"Option {index}." for index in range(option_count)}
            self({}, {"warmup": {"type": "choice", "instructions": "Pick one.", "criteria": criteria}})


def run_directory(mode):
    return RUNS_DIR / ("w8a8" if MODES[mode] else "bf16-activations")


for mode, int8_activations in MODES.items():
    started = time.perf_counter()
    summary = run("__main__:ClefTpuEngine", {"int8_activations": int8_activations}, ROWS_PATH, run_directory(mode), log=lambda line: None)
    print(f"{mode}: {summary['counts']} in {time.perf_counter() - started:.0f}s")

## Results

Accuracy counts unanswered or failed requests as wrong, as the Decision Index does. The ± is the 95% interval from sampling 1,172 questions: a gap to 98.3 smaller than that is noise. Cloudflare's latency was measured on an H200 GPU across the whole Decision Index, so it is not directly comparable.

In [ ]:
import collections
import importlib.metadata
import math
import statistics

from decision_index.suite.io import read_jsonl

CLOUDFLARE_ARC_CHALLENGE = 98.3


def percentile(values, percent):
    return statistics.quantiles(values, n=100, method="inclusive")[percent - 1]


def mode_results(mode):
    results = {row["run_id"]: row for row in read_jsonl(run_directory(mode) / "results.jsonl")}
    assert results.keys() == expected_choices.keys(), f"{mode}: {len(results)} of {len(expected_choices)} requests have results"
    return results


def chosen(result):
    return result["response"]["answers"]["q1"]["choice"] if result["status"] == "ok" else None


CARD_LABELS = {False: "This port, int8 weights (default)", True: "This port, W8A8 (`int8_activations=True`)"}
TESTED_PACKAGES = ("jax", "jaxlib", "transformers", "safetensors", "torch", "huggingface_hub")

results_by_mode = {mode: mode_results(mode) for mode in MODES}
summaries = {}
for mode, results in results_by_mode.items():
    statuses = collections.Counter(result["status"] for result in results.values())
    latencies = [result["total_wall_ms"] for result in results.values() if result["status"] == "ok"]
    summaries[mode] = {
        "correct": sum(chosen(result) == expected_choices[run_id] for run_id, result in results.items()),
        "median_ms": statistics.median(latencies),
        "p95_ms": percentile(latencies, 95),
    }
    print(f"{mode}: {summaries[mode]['correct']}/{len(results)} correct, statuses {dict(statuses)}")

question_count = len(expected_choices)
default_mode, w8a8_mode = MODES
default, w8a8 = summaries[default_mode], summaries[w8a8_mode]
default_accuracy = default["correct"] / question_count
margin_points = 100 * 1.96 * math.sqrt(default_accuracy * (1 - default_accuracy) / question_count)
is_within_margin = abs(100 * default_accuracy - CLOUDFLARE_ARC_CHALLENGE) <= margin_points
cloudflare_correct = round(CLOUDFLARE_ARC_CHALLENGE / 100 * question_count)
changed_answers = sum(chosen(results_by_mode[default_mode][run_id]) != chosen(results_by_mode[w8a8_mode][run_id]) for run_id in expected_choices)
median_tokens = statistics.median(result["response"]["usage"]["input_tokens"] for result in results_by_mode[default_mode].values() if result["status"] == "ok")


def compared_to_original(correct):
    gap = correct - cloudflare_correct
    if gap == 0:
        return "the same number of questions as"
    return f"{abs(gap)} question{'' if abs(gap) == 1 else 's'} {'more' if gap > 0 else 'fewer'} than"


table_rows = "\n".join(
    f"| {CARD_LABELS[MODES[mode]]} | {summary['correct']:,} | {100 * summary['correct'] / question_count:.1f}% | "
    f"{summary['median_ms']:.0f} ms | {summary['p95_ms']:.0f} ms |"
    for mode, summary in summaries.items()
)
version_rows = "\n".join(f"| `{package}` | {importlib.metadata.version(package)} |" for package in TESTED_PACKAGES)

results_section = f"""### ARC-Challenge

{question_count:,} grade-school science questions from the [Decision Index]({KIT_URL}) (benchmark 27), the suite behind Cloudflare's published results, scored the same way.

| Model | Correct | Accuracy | Median latency | p95 latency |
|---|---:|---:|---:|---:|
| Cloudflare clef-flash, BF16 (published) | ≈{cloudflare_correct:,} | {CLOUDFLARE_ARC_CHALLENGE}% | | |
{table_rows}

- **int8 {'matches' if is_within_margin else 'differs from'} the original:** it answers {compared_to_original(default['correct'])} Cloudflare's BF16 model, {'within' if is_within_margin else 'outside'} the ±{margin_points:.1f}-point sampling margin for {question_count:,} questions.
- **W8A8** is {100 * (1 - w8a8['median_ms'] / default['median_ms']):.0f}% faster and changes the answer on {changed_answers} of {question_count:,} questions.
- **Latency** is one request at a time on one {DEVICE_KIND} chip (Colab's v5e-1). ARC prompts (median {median_tokens:.0f} tokens) all fit the first 1,024-token bucket. Cloudflare's published latency comes from an H200 GPU across the whole suite, so it isn't comparable.

<details>
<summary>Versions and sources</summary>

| Package | Version |
|---|---|
{version_rows}

Rows built with [decision-index]({KIT_URL}) @ `{KIT_COMMIT[:12]}` from `{ARC_REPO}` @ `{ARC_REVISION[:12]}`; weights @ `{MODEL_REVISION[:12]}`.

</details>"""
print(results_section)

RESULTS_START, RESULTS_END = "<!-- arc-challenge-results:start -->", "<!-- arc-challenge-results:end -->"
readme = Path(hf_hub_download(MODEL_REPO, "README.md", force_download=True)).read_text()
assert readme.count(RESULTS_START) == 1 and readme.count(RESULTS_END) == 1, "README.md needs exactly one pair of arc-challenge-results markers"
before_section, rest = readme.split(RESULTS_START)
_, after_section = rest.split(RESULTS_END)
commit = HfApi().upload_file(
    path_or_fileobj=f"{before_section}{RESULTS_START}\n{results_section}\n{RESULTS_END}{after_section}".encode(),
    path_in_repo="README.md",
    repo_id=MODEL_REPO,
    commit_message=f"ARC-Challenge results for weights {MODEL_REVISION[:12]}",
)
print(f"model card updated: {commit.commit_url}")

## Notes

- **The rows match the kit's ARC builder field for field,** but come from the current `allenai/ai2_arc` revision rather than the kit's pinned copy. The printed revision records which one ran.
- **Other benchmarks:** rebuild the full suite with the kit (`suite rebuild`, about 7 GB of downloads) and run the same `ClefTpuEngine` over its rows.
- **Truncation:** `clef_tpu`, like Cloudflare's code, cuts states longer than 16,384 tokens, and the kit's rules forbid that. ARC prompts are far shorter, so it doesn't matter here; a full-suite run would need the engine to raise `Unsupported` for those requests instead.